In [ ]:
# (DO NOT containerize this cell)
# Data handling parameters
param_minio_endpoint = 'scruffy.lab.uvalight.net:9000'
param_minio_virtual_lab_bucket = 'naa-vre-laserfarm'
param_files_to_process_file = 'AHN6_url_veluwe.txt'
param_max_tiles = 4
param_max_nodes = 2

In [ ]:
# Secrets (DO NOT containerize this cell)
from SecretsProvider import SecretsProvider
from getpass import getpass

secrets_provider = SecretsProvider(input_func=getpass)
secret_minio_access_key = secrets_provider.get_secret('secret_minio_access_key')
secret_minio_secret_key = secrets_provider.get_secret('secret_minio_secret_key')

In [ ]:
# (DO NOT containerize this cell)
import random
import math
import warnings
import requests
import json
import io

from minio import Minio
from minio.error import S3Error
from typing import List
from urllib import request
from pathlib import Path

In [ ]:
# (DO NOT containerize this cell)
conf_minio_path_to_process = 'toProcess'
conf_minio_path_to_download = 'toDownload'
conf_minio_path_original = 'original'

In [ ]:
# datafiles batch creator
def get_filename_batches_to_process() -> List[str]:
    filenames_to_process = get_minio_file_as_set(f"{conf_minio_path_to_process}/{param_files_to_process_file}", fail_if_file_not_found=True)
    if not filenames_to_process:
        raise ValueError(f"Found no files to process.")
    unprocessed_filenames = keep_unprocessed_urls(filenames_to_process)
    if not unprocessed_filenames:
        raise ValueError(f"The list of unprocessed_filenames is empty. Found {len(filenames_to_process)} files which have alreay been processed.")
    random.shuffle(unprocessed_filenames) 
    selected_filenames = unprocessed_filenames[:param_max_tiles]
    workflow_files = save_lists_to_cloud_storage(batches=param_max_nodes, folder=conf_minio_path_to_download, file_basepath=f"{conf_minio_path_to_download}.yaml", filenames=selected_filenames)
    print(f"Files to process: {len(filenames_to_process)}, of which {len(unprocessed_filenames)} are unprocessed. Preparing {len(selected_filenames)} files in {len(workflow_files)} batches.")
    return workflow_files

def get_minio_file_as_set(filepath: str, fail_if_file_not_found: bool) -> set[str]:
    response = None
    try:
        response = minio_client.get_object(param_minio_virtual_lab_bucket, filepath)
        content = response.data.decode("utf-8")
        return {line.strip() for line in content.splitlines() if line.strip()}

    except S3Error as e:
        if ((not fail_if_file_not_found) and e.code == "NoSuchKey"):
            warnings.warn(f"No file found in path: {filepath}.")
            return set()
        raise e
        
    finally:
        if response:
            response.close()
            response.release_conn()
    
def keep_unprocessed_urls(urls: List[str]) -> List[str]:
    objects = minio_client.list_objects(param_minio_virtual_lab_bucket, prefix=conf_minio_path_original, recursive=True)
    processed_filenames = {str(Path(obj.object_name).name) for obj in objects}
    files_to_process = [url for url in urls if Path(url).name not in processed_filenames]
    return files_to_process

def save_lists_to_cloud_storage(batches: int, folder: str, file_basepath: str, filenames: List[str]) -> List[str]:
    filename_batches = split_into_batches(batches, filenames)
    batch_filenames = []
    for n, batch in enumerate(filename_batches):
        filepath = f"{folder}/{n}.{file_basepath}"
        json_batch = json.dumps(batch)
        save_json_in_cloud_storage(filepath, json_batch)
        batch_filenames.append(filepath)
    return batch_filenames

def split_into_batches(max_batches: int, filenames: List[str]):
    batch_size = math.ceil(len(filenames)/max_batches)
    return[filenames[i:i + batch_size] for i in range(0, len(filenames), batch_size)]
    
def save_json_in_cloud_storage(filepath: str, json) -> str:
    json_file = io.BytesIO(json.encode("utf-8"))
    minio_client.put_object(bucket_name=param_minio_virtual_lab_bucket, object_name=str(filepath), data=json_file, length=len(json), content_type="application/json")

minio_client = Minio(
    param_minio_endpoint, 
    access_key=secret_minio_access_key,
    secret_key=secret_minio_secret_key,
    secure=True
)

las_data_batch_files: List[str] = get_filename_batches_to_process()

In [ ]:
# Retrieve laz files
def read_file_list_from_json(json_file_path: str) -> List[str]:
    response = minio_client.get_object(param_minio_virtual_lab_bucket, json_file_path)
    file_data = response.read()
    return json.loads(file_data.decode("utf-8"))

def fetch_and_upload(url: str) -> None:
    with requests.get(url, stream=True) as response:
        response.raise_for_status() 
        file_size = int(response.headers.get('Content-Length', 0))
        filepath = Path(conf_minio_path_original) / Path(url).name
        minio_client.put_object(bucket_name=param_minio_virtual_lab_bucket, object_name=str(filepath), data=response.raw, length=file_size)

minio_client = Minio(
    param_minio_endpoint, 
    access_key=secret_minio_access_key,
    secret_key=secret_minio_secret_key,
    secure=True
)
    
for batch_file in las_data_batch_files:
    batch = read_file_list_from_json(batch_file)
    for url in batch:
        fetch_and_upload(url)

downloaded_file_folder: str = conf_minio_path_original

In [ ]:
# (DO NOT containerize this cell)
# Dummy use output
print(downloaded_file_folder)